In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from configs.config import JSON_REPORTS_DIR, JSON_CHUNKS_DIR

# --- Papermill parameters (overwritten at runtime) ---
input_dir   = str(JSON_REPORTS_DIR)
output_file = str(JSON_CHUNKS_DIR / "rag_chunks_all.jsonl")
run_id      = "rag_chunks_all"
experiment_name = "baseline"

In [ ]:
# Incluir esto al comienzo del notebook (después de la celda de parámetros si usas papermill)
import mlflow

# Asegurar que estamos en el run correcto sin iniciar uno nuevo
if mlflow.active_run() is None and "run_id" in globals():
    mlflow.start_run(run_id=run_id)

In [ ]:
import os
import json

from src.utils import table_to_markdown, split_into_chunks

MAX_CHARS_PER_CHUNK = 1000  # ~1-2K works well for most embedding models

os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f_out:
    for filename in os.listdir(input_dir):
        if not filename.endswith("_by_page.json"):
            continue

        input_file = os.path.join(input_dir, filename)
        doc_id     = filename.replace("_by_page.json", "")

        with open(input_file, "r", encoding="utf-8") as f_in:
            pages = json.load(f_in)

        for page_data in pages:
            page   = page_data.get("page", 0)
            text   = page_data.get("text", "").strip()
            tables = page_data.get("tables", [])

            table_texts = [table_to_markdown(tbl) for tbl in tables]
            combined    = text + ("\n\n" + "\n\n".join(table_texts) if table_texts else "")
            combined    = combined.strip()

            if not combined:
                continue

            for idx, chunk in enumerate(split_into_chunks(combined, MAX_CHARS_PER_CHUNK)):
                json.dump({
                    "doc_id":   doc_id,
                    "page":     page,
                    "chunk_id": f"{page}_{idx}",
                    "content":  chunk,
                    "source":   input_file,
                }, f_out, ensure_ascii=False)
                f_out.write("\n")

print(f"Chunks saved to: {output_file}")

In [ ]:
import json
import mlflow

total_chars = 0
num_lines = 0

with open(output_file, "r", encoding="utf-8") as f:
    for line in f:
        num_lines += 1
        total_chars += len(json.loads(line).get("content", "").strip())

print(f"Total chunks:     {num_lines}")
print(f"Total characters: {total_chars:,}")
print(f"Mean chars/chunk: {round(total_chars / num_lines):,}")

mlflow.log_metrics({
    "total_chunks":      num_lines,
    "total_characters":  total_chars,
    "mean_chars_per_chunk": round(total_chars / num_lines),
})